# B5 — Parameter Estimation

**Reference:** West & Harrison §4.4, §10; Petris §3

**Concepts introduced:**
- V and W are rarely known in practice — we must estimate them from data
- MLE via `scipy.optimize.minimize` on the Kalman filter log-likelihood
- 2D likelihood surface — the ridge shaped by identifiability
- Bayesian estimation with PyMC (blackbox likelihood via `as_op`)
- Comparing MLE point estimate to posterior uncertainty

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from engine.models import make_local_level
from engine.filter import kalman_filter
from engine.simulate import simulate

## 1. The estimation problem

In the local level model, $V$ and $W$ are unknown. We need to estimate them.

**Two approaches:**

| Approach | Tool | Returns |
|----------|------|---------|
| **MLE** | `scipy.optimize.minimize` | point estimate $(\hat V, \hat W)$ |
| **Bayesian** | PyMC + Kalman blackbox likelihood | full posterior $p(V, W \mid y)$ |

Both use the `.loglik` attribute of the `FilterResult` returned by `kalman_filter(spec, y)` as the objective / likelihood.

In [ ]:
# Simulate data
V_true, W_true = 2.0, 0.5
spec_true = make_local_level(V=V_true, W_level=W_true)
sim = simulate(spec_true, n=100, seed=0)
y = sim.y    # (100, 1)

## 2. MLE via scipy.optimize

We maximise $\log p(y \mid V, W)$ by minimising its negation.

**Log-space parameterisation:** we optimise $\log V$ and $\log W$ instead of
$V$ and $W$ directly. This enforces positivity and improves numerical conditioning —
$\log V$ ranges over all of $\mathbb R$ while $V > 0$.

In [ ]:
def neg_loglik(params: np.ndarray) -> float:
    V_opt, W_opt = np.exp(params)
    spec = make_local_level(V=V_opt, W_level=W_opt)
    return -kalman_filter(spec, y).loglik

result = minimize(neg_loglik, x0=np.array([0.0, 0.0]), method="Nelder-Mead",
                  options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 2000})
V_mle, W_mle = np.exp(result.x)

print(f"Converged: {result.success}")
print(f"V_mle = {V_mle:.4f}  (true: {V_true})")
print(f"W_mle = {W_mle:.4f}  (true: {W_true})")
print(f"\u03ba_mle = W/V = {W_mle/V_mle:.4f}  (true: {W_true/V_true:.4f})")

## 3. Likelihood surface

The 2D likelihood surface $\log p(y \mid V, W)$ has a characteristic **ridge**
shape: many (V, W) pairs with the same ratio $W/V$ give nearly identical likelihoods.
This reflects partial non-identifiability — only the ratio is well-determined from
a single series of modest length.

In [ ]:
V_grid = np.exp(np.linspace(-1, 2.5, 50))
W_grid = np.exp(np.linspace(-2.5, 1, 50))
LL = np.zeros((len(V_grid), len(W_grid)))

for i, V_try in enumerate(V_grid):
    for j, W_try in enumerate(W_grid):
        sp = make_local_level(V=V_try, W_level=W_try)
        LL[i, j] = kalman_filter(sp, y).loglik

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(np.log(W_grid), np.log(V_grid), LL, levels=40, cmap="viridis")
fig.colorbar(cf, ax=ax, label="log p(y | V, W)")
ax.plot(np.log(W_mle), np.log(V_mle), "r*", ms=12, label="MLE")
ax.plot(np.log(W_true), np.log(V_true), "w+", ms=12, mew=2, label="true")
ax.set(xlabel="log W", ylabel="log V", title="Log-likelihood surface")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Bayesian estimation with PyMC

The Kalman filter is pure NumPy — PyTensor (PyMC's backend) cannot differentiate
through it. We register the log-likelihood function as a **blackbox op** via
`pytensor.compile.ops.as_op`. This tells PyMC to use the **Slice sampler**
(gradient-free) instead of NUTS (requires gradients).

**Note:** The Slice sampler is less efficient than NUTS for smooth posteriors but
works correctly here. For production use, consider implementing the filter in
PyTensor for NUTS compatibility.

In [ ]:
HAS_PYMC = False
try:
    import pymc as pm
    import pytensor.tensor as pt
    import pytensor.compile.ops as ops
    import arviz as az
    HAS_PYMC = True
except ImportError:
    print("PyMC not installed \u2014 skipping Bayesian estimation cells")

if HAS_PYMC:
    @ops.as_op(itypes=[pt.dscalar, pt.dscalar], otypes=[pt.dscalar])
    def kalman_loglik_op(V_val: np.ndarray, W_val: np.ndarray) -> np.ndarray:
        spec = make_local_level(V=float(V_val), W_level=float(W_val))
        return np.array(kalman_filter(spec, y).loglik)

    with pm.Model() as dlm_model:
        V_rv = pm.HalfNormal("V", sigma=3.0)
        W_rv = pm.HalfNormal("W", sigma=1.0)
        _    = pm.Potential("loglik", kalman_loglik_op(V_rv, W_rv))
        idata = pm.sample(500, tune=500, progressbar=False, random_seed=42)

    print(az.summary(idata, var_names=["V", "W"]))

The plot below shows the full posterior distribution for $V$ and $W$, overlaid
with the MLE point estimate (dashed red) and the true value (solid black).

**What to look for:**
- Is the true value inside the bulk of the posterior?
- How wide is each posterior? Width reflects how much information the data carry.
- The MLE should sit near the posterior mode for weakly informative priors.

In [ ]:
if HAS_PYMC:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, var, true_val, mle_val in zip(
        axes,
        ["V", "W"],
        [V_true, W_true],
        [V_mle, W_mle],
    ):
        samples = idata.posterior[var].values.ravel()
        ax.hist(samples, bins=50, density=True, alpha=0.6, color="C1", label="Posterior")
        ax.axvline(true_val, color="k", ls="-",  lw=2, label=f"True {var}={true_val}")
        ax.axvline(mle_val,  color="r", ls="--", lw=2, label=f"MLE {var}={mle_val:.2f}")
        ax.set(xlabel=var, ylabel="density", title=f"Posterior vs MLE: {var}")
        ax.legend()
    plt.tight_layout()
    plt.show()

## Exercises

**Exercise 1** — Simulate a short series (T=20) and a long series (T=200) with
the same V=2.0, W=0.5. Run MLE on both. How does estimation error change with T?

**Exercise 2** *(requires PyMC; skip if unavailable)* — Replace `HalfNormal` priors
with `pm.Exponential("V", lam=0.5)` and `pm.Exponential("W", lam=1.0)`. Re-run.
How does the posterior change, especially for the short T=20 series?

**Exercise 3** — Plot the likelihood profile over V with W fixed at W_mle
(a 1D slice of the surface). Mark the MLE and the posterior mean. Are they close?